# Süzgeç — turkish-mini-bert deneyi (derin öğrenme)
**Ne yapar?** `ytu-ce-cosmos/turkish-mini-bert-uncased` modelini (Kesgin ve ark., 2023; 11,6M parametre) Süzgeç'in 11 içerik türü için ince ayarlar ve uygulamadaki **TF-IDF + lojistik regresyon** modeliyle **aynı eğitim/test bölmesinde** karşılaştırır.

**Çalıştırma:** 1) Çalışma zamanı → Çalışma zamanı türünü değiştir → **T4 GPU**. 2) Soldaki klasör simgesinden `suzgec_bert_veri.csv` dosyasını yükle. 3) Çalışma zamanı → **Tümünü çalıştır**. ~10–15 dk sürer.
4) Sonda oluşan `bert_sonuc.json` ve `bert_karsilastirma.png` dosyalarını indir (sağ tık → İndir).

In [ ]:
!pip -q install transformers==4.* accelerate datasets scikit-learn pandas
import os, re, json, time, numpy as np, pandas as pd, torch
from sklearn.metrics import f1_score, recall_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
print('GPU:', torch.cuda.is_available())
SEED=26; np.random.seed(SEED); torch.manual_seed(SEED)

In [ ]:
d = pd.read_csv('suzgec_bert_veri.csv', keep_default_na=False)
TURLER = {1:'guvenlik_para',2:'kisisel_mesaj',3:'grup_mesaji',4:'okul_is_resmi',5:'zaman_hassas',6:'islem_bilgi',
          7:'kampanya',8:'kumar',9:'sosyal_etkilesim',10:'geri_cagirma',11:'sistem_diger'}
URL_RE=re.compile(r"(https?://\S+|www\.\S+|\b[a-z0-9-]+\.(?:ly|com|link|su|me|net|org|tr|io|co)(?:/\S*)?)")
TEL_RE=re.compile(r"(\+?90[\s-]?)?0?5\d{2}[\s-]?\d{3}[\s-]?\d{2}[\s-]?\d{2}")
NUM_RE=re.compile(r"\d[\d.,]{3,}"); WS=re.compile(r"\s+")
def tr_lower(s): return s.replace('İ','i').replace('I','ı').lower()
def on_isle(a,t,x, katla):   # katla=True: uygulamadaki v2 (ASCII katlama); BERT için False
    s=tr_lower(f"{a} | {t} | {x}"); s=URL_RE.sub(' <url> ',s); s=TEL_RE.sub(' <tel> ',s); s=NUM_RE.sub(' <num> ',s)
    if katla: s=s.translate(str.maketrans('şığüöç','siguoc'))
    return WS.sub(' ',s).strip()
d['x_lr']=[on_isle(a,t,x,True) for a,t,x in zip(d.app,d.title,d.text)]
d['x_bert']=[on_isle(a,t,x,False) for a,t,x in zip(d.app,d.title,d.text)]
tr=d[d.split=='train']; print(d.split.value_counts())

In [ ]:
def olc(y,p,name):
    y=np.asarray(y); p=np.asarray(p)
    r={'test':name,'n':len(y),'makroF1':round(f1_score(y,p,average='macro'),3),
       'kampanya_duyarlilik':round(recall_score(y,p,labels=[7],average='macro',zero_division=0),3),
       'kampanya->kumar':int(((y==7)&(p==8)).sum()),
       'guvenlik_duyarlilik':round(recall_score(y,p,labels=[1],average='macro',zero_division=0),3) if (y==1).any() else None,
       'kumar_duyarlilik':round(recall_score(y,p,labels=[8],average='macro',zero_division=0),3) if (y==8).any() else None,
       'kisisel->silinen':int(((y==2)&((p==7)|(p==8))).sum())}
    return r
# Taban: uygulamadaki TF-IDF + LR (aynı ayarlar)
lr=make_pipeline(TfidfVectorizer(analyzer='char_wb',ngram_range=(2,5),sublinear_tf=True,lowercase=False,min_df=2,max_features=30000),
                 LogisticRegression(C=4.0,max_iter=3000,class_weight='balanced')).fit(tr.x_lr,tr.label)
sonuc={'TFIDF_LR':[olc(d[d.split==s].label,lr.predict(d[d.split==s].x_lr),s) for s in ['test_uysal','test_elle']]}
sonuc

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
MID='ytu-ce-cosmos/turkish-mini-bert-uncased'
tok=AutoTokenizer.from_pretrained(MID)
lab2id={k:i for i,k in enumerate(sorted(TURLER))}; id2lab={i:k for k,i in lab2id.items()}
model=AutoModelForSequenceClassification.from_pretrained(MID,num_labels=11,id2label={i:TURLER[k] for i,k in id2lab.items()},
      label2id={TURLER[k]:i for k,i in lab2id.items()})
print('parametre:', sum(p.numel() for p in model.parameters())/1e6, 'M')
def ds(df): 
    e=tok(list(df.x_bert),truncation=True,max_length=64)
    e['labels']=[lab2id[l] for l in df.label]; return Dataset.from_dict(dict(e))
# sınıf ağırlıkları (dengesiz veri)
cnt=tr.label.value_counts(); w=torch.tensor([len(tr)/(11*cnt[id2lab[i]]) for i in range(11)],dtype=torch.float)
class WTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels=inputs.pop('labels'); out=model(**inputs)
        loss=torch.nn.functional.cross_entropy(out.logits,labels,weight=w.to(out.logits.device))
        return (loss,out) if return_outputs else loss
args=TrainingArguments('out',per_device_train_batch_size=64,num_train_epochs=4,learning_rate=5e-5,weight_decay=0.01,
     warmup_ratio=0.06,logging_steps=50,save_strategy='no',report_to=[],seed=SEED,fp16=torch.cuda.is_available())
from transformers import DataCollatorWithPadding
t0=time.time(); WTrainer(model=model,args=args,train_dataset=ds(tr),data_collator=DataCollatorWithPadding(tok)).train()
print('eğitim süresi (sn):', round(time.time()-t0))

In [ ]:
@torch.no_grad()
def bert_tahmin(texts, bs=128, device=None):
    device=device or ('cuda' if torch.cuda.is_available() else 'cpu'); model.to(device).eval(); P=[]
    for i in range(0,len(texts),bs):
        e=tok(texts[i:i+bs],truncation=True,max_length=64,padding=True,return_tensors='pt').to(device)
        P.append(torch.softmax(model(**e).logits,-1).cpu().numpy())
    return np.vstack(P)
sonuc['turkish_mini_bert']=[]
for s in ['test_uysal','test_elle']:
    df=d[d.split==s]; p=bert_tahmin(list(df.x_bert)).argmax(1)
    sonuc['turkish_mini_bert'].append(olc(df.label,[id2lab[i] for i in p],s))
# CPU gecikmesi (telefona yakın ölçüm için tek örnek, tek iş parçacığı)
torch.set_num_threads(1); model.to('cpu'); x=list(d[d.split=='test_elle'].x_bert[:100])
t0=time.time(); [bert_tahmin([s],device='cpu') for s in x]; bert_ms=(time.time()-t0)/len(x)*1000
t0=time.time(); [lr.predict_proba([s]) for s in d[d.split=='test_elle'].x_lr[:100]]; lr_ms=(time.time()-t0)/100*1000
sonuc['gecikme_ms_cpu_tek_ornek']={'turkish_mini_bert':round(bert_ms,2),'TFIDF_LR':round(lr_ms,2)}
sonuc['boyut_MB']={'turkish_mini_bert':round(sum(p.numel() for p in model.parameters())*4/1e6,1),'TFIDF_LR_json':3.86}
print(json.dumps(sonuc,ensure_ascii=False,indent=1))

In [ ]:
import matplotlib.pyplot as plt
rows=[]
for m in ['TFIDF_LR','turkish_mini_bert']:
    for r in sonuc[m]: rows.append({'model':m,**r})
t=pd.DataFrame(rows); display(t)
fig,ax=plt.subplots(1,2,figsize=(10,3.5))
for i,s in enumerate(['test_uysal','test_elle']):
    x=t[t.test==s].set_index('model')[['makroF1','kampanya_duyarlilik','guvenlik_duyarlilik']]
    x.T.plot.bar(ax=ax[i],rot=0,ylim=(0,1.05),title={'test_uysal':'Uysal 2013 SMS (772, eğitimde yok)','test_elle':'Elle yazılmış bildirimler (138)'}[s])
plt.tight_layout(); plt.savefig('bert_karsilastirma.png',dpi=160)
json.dump(sonuc,open('bert_sonuc.json','w'),ensure_ascii=False,indent=1)
print(classification_report(d[d.split=='test_elle'].label,[id2lab[i] for i in bert_tahmin(list(d[d.split=='test_elle'].x_bert)).argmax(1)],zero_division=0))

**İsteğe bağlı:** modeli kaydetmek için şu hücreyi çalıştır (telefona taşımak için ONNX'e çevrilebilir; ~47 MB).

In [ ]:
# model.save_pretrained('suzgec_mini_bert'); tok.save_pretrained('suzgec_mini_bert')
# !zip -qr suzgec_mini_bert.zip suzgec_mini_bert